# Notebook 1 — Market Survey

**Question:** what does the modern Pokémon singles market actually look like, and where (if anywhere) is there edge for a $100 bankroll?

**Method:** model buying at the lowest current listing and selling at market price, then apply the fee model from `pokedb/fees.py` to see what survives.

**Read the reality check in §4 before acting on anything here.** The screen is deliberately optimistic, and the honest answer it produces is not the encouraging one.

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from pokedb import analysis, db, fees

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Your eBay final value fee. Retail is 13.25%.
# OVERRIDE ON AUG 17 with the real employee rate.
YOUR_RATE = 0.05

conn = db.connect()
print(f"snapshot days collected: {analysis.history_depth(conn)}")

## 1. The tradeable universe

Before looking for opportunities, establish how much of the catalog can be traded at all.

In [ ]:
bands = analysis.price_band_summary(conn)
bands["share"] = (bands.cards / bands.cards.sum()).map("{:.1%}".format)
bands

**Roughly three quarters of the modern catalog is worth under $1.**

Shipping a single card costs $1.25 in a tracked plain white envelope, before the per-order fee, before supplies, before the platform takes its cut. Cards below about $3 cannot be sold individually at a profit at *any* fee rate. They are bulk — sold by the thousand, by weight, at a few cents each — not inventory.

That is why `analysis.BULK_CEILING` is $3. Everything below it is excluded from here on, not because it is uninteresting but because it is not a product.

In [ ]:
df = analysis.add_edge_columns(
    analysis.load_tradeable(conn, min_price=analysis.BULK_CEILING, max_price=40.0),
    your_rate=YOUR_RATE,
)
print(f"{len(df):,} candidate cards in the ${analysis.BULK_CEILING:.0f}-40 band")
df[["name", "set_abbr", "rarity", "low_price", "market_price", "spread_pct"]].head()

## 2. The screen: who can profitably make each trade?

`breakeven_fvf` gives the final value fee rate at which a trade breaks even. Comparing it to your rate and to retail's 13.25% sorts every card into three zones:

| Zone | Meaning |
| --- | --- |
| `open` | Profitable at retail rates too. Crowded. |
| `exclusive` | Profitable for you, not for a retail flipper. The thesis says hunt here. |
| `dead` | Unprofitable at any accessible rate. |

In [ ]:
analysis.edge_zone_summary(df)

The `clearing_threshold` column counts cards netting at least $1.50 — the floor worth the sourcing, listing, packing, and return risk.

**The exclusive zone is real but thin.** It holds a few hundred cards, but its median profit is well under a dollar and only a few dozen clear the handling threshold. The thesis is directionally right and much smaller than hoped.

## 3. What the fee discount is actually worth

In [ ]:
print(f"median card in band:     ${df.market_price.median():.2f}")
print(f"median fee advantage:    ${df.fee_advantage.median():.2f} per trade")
print(f"  as % of median card:   {df.fee_advantage.median() / df.market_price.median():.1%}")
print()

bankroll, avg_card = 100.0, df.market_price.median()
n = int(bankroll // avg_card)
print(f"${bankroll:.0f} buys ~{n} cards at the median price")
print(f"total edge from the fee discount alone: ~${n * df.fee_advantage.median():.2f}")

Worth internalising: the discount is a genuine structural advantage, and across a $100 bankroll it amounts to single-digit dollars. It improves the odds on trades you were going to make. It does not by itself turn a losing strategy into a winning one.

## 4. Reality check — why the top of the screen is mostly traps

The biggest apparent opportunities look extraordinary. That is the problem.

In [ ]:
df.nlargest(10, "profit_at_your_rate")[
    ["name", "set_abbr", "low_price", "market_price", "spread_pct",
     "profit_at_your_rate", "edge_zone"]
]

A card with a $30 market price listed at $9 is not a gift somebody forgot about. It is one of:

- a **damaged or heavily played copy** — TCGCSV publishes no condition, so a DMG copy and a NM copy look identical here
- a **misidentified print** — a different set, a reprint, or a non-English copy
- a **listing that has already sold** and not yet cleared
- an **outright error or scam**

Note also that the largest opportunities are almost all `open` zone — profitable at retail too. If a trade is genuinely that good and visible to everyone, it has already been taken. Its survival on the screen is itself evidence something is wrong with it.

`spread_is_suspicious` flags spreads over 50%.

In [ ]:
print(f"suspicious spreads: {df.spread_is_suspicious.sum():,} of {len(df):,}")
print()
print("Median profit by rarity — buying at low, selling at market:")
analysis.rarity_summary(df)

**The median profit is negative in every rarity.**

That is the honest headline of this notebook. The typical card in the tradeable band loses money on a buy-at-low, sell-at-market round trip once fees and shipping are paid. The market is efficient at this size, exactly as expected for the most liquid, most-watched corner of the hobby.

There is a deeper problem with this screen, and it is structural rather than a matter of filtering harder:

> **Buying at TCGplayer low and selling at TCGplayer market is not an arbitrage.** It is the same order book. Relisting a card into the venue you just bought it from puts you behind the same low listings you were competing with, and market price is a trailing average of sales that already happened — not a price anyone owes you.

Real edge has to come from **a price difference between two venues**, where one side is genuinely less efficient. That is eBay, and specifically the parts of eBay where cards are hard to find: bad titles, missing set names, misspellings, auctions ending at 3am, sellers who list a $20 card as "pokemon card rare holo".

**This finding raises the priority of Phase 3 (eBay Browse ingestion + title parsing).** It is not a nice-to-have layered on top of the TCGplayer data — without a second venue there is no arbitrage to measure at all.

## 5. Watchlist

Even given the above, this survey produces something useful: a set of cards worth *monitoring* on the other venue. These are cards where dispersion is wide enough that a genuine mispricing could exist, without being so wide that it is obviously an artifact.

The watchlist is not a buy list. It is the universe Phase 3 will search eBay for.

In [ ]:
cand = analysis.candidates(df, min_profit=analysis.MIN_WORTHWHILE_PROFIT)
print(f"{len(cand)} candidates after removing suspicious spreads\n")
cand.head(20)[
    ["name", "set_abbr", "sub_type_name", "rarity", "low_price",
     "market_price", "profit_at_your_rate", "edge_zone"]
]

In [ ]:
n = analysis.write_watchlist(conn, cand)
print(f"wrote {n} cards to the watchlist")
print("\ntarget_buy_price is set 15% below the current low — paying the visible")
print("low price captures nothing, because that price is already public.")

pd.read_sql(
    """SELECT p.name, s.abbreviation AS set_abbr, w.target_buy_price
       FROM watchlist w
       JOIN products p ON p.product_id = w.product_id
       JOIN sets s ON s.group_id = p.group_id
       ORDER BY w.target_buy_price DESC LIMIT 10""",
    conn,
)

## 6. What this notebook cannot answer yet

Everything above is computed from a **single day's snapshot**. That is enough for structure — how the catalog distributes, where dispersion sits, what the fee model implies — and not enough for anything dynamic:

- **Volatility.** Which cards actually move, and by how much.
- **Trend.** Whether a wide spread is a standing feature of a card or today's noise.
- **Liquidity.** How fast prices update, which is the closest available proxy for whether anything is trading at all.

All three need a run of daily snapshots, and TCGCSV cannot be backfilled — it serves only the current day. Every day the ingest does not run is a day of history that will never exist.

```
0 22 * * *  cd /path/to/Learning && python3 scripts/ingest_tcgcsv.py --quiet >> ingest.log 2>&1
```

Revisit this notebook once there are two weeks of data.